# VIScan on a GPU runtime

This notebook is one place to run the model. The laptop app does not know it is Colab. It only calls the URL printed at the end. The same server (`scripts/model_server.py`) runs on Kaggle or another machine.

Runs the VIA vision model on this GPU. The laptop app stays where it is and sends each image here.

Before the first code cell: **Runtime → Change runtime type → T4 GPU**.

On the laptop, zip `ai-interpreter` without `.venv` and without `models` (Colab downloads the weights itself):

```bash
cd ai-interpreter
zip -r ~/ai-interpreter.zip app scripts notebooks run.py requirements.txt requirements-model.txt requirements-dev.txt
```

The last cell prints three lines. Paste them into `ai-interpreter/.env` on the laptop and restart `python run.py`. A free Colab session stops when it is idle, and the URL is new each time you start again.

In [ ]:
import torch
print('cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no GPU')
if not torch.cuda.is_available():
    raise SystemExit('Turn on a GPU runtime, then run this cell again.')

In [ ]:
import os, zipfile
from pathlib import Path

def find_root() -> Path:
    for folder in Path('/content').rglob('interpreter.py'):
        if folder.parent.name == 'services' and folder.parent.parent.name == 'app':
            return folder.parents[2]
    raise FileNotFoundError('interpreter.py was not in the upload')

try:
    ROOT = find_root()
    print('Using', ROOT)
except FileNotFoundError:
    from google.colab import files
    print('Choose ai-interpreter.zip')
    uploaded = files.upload()
    name = next(iter(uploaded))
    with zipfile.ZipFile(name) as archive:
        archive.extractall('/content')
    ROOT = find_root()
    print('Using', ROOT)
os.chdir(ROOT)

In [ ]:
!pip install -q -r requirements.txt -r requirements-model.txt
!python scripts/pull_model.py

In [ ]:
import re, secrets, subprocess, time, urllib.request
from pathlib import Path

token = secrets.token_hex(16)
port = 8088
log_path = Path('/content/viscan-colab.log')
log = log_path.open('w')
server = subprocess.Popen(
    ['python', 'scripts/model_server.py', '--port', str(port), '--token', token],
    stdout=log, stderr=subprocess.STDOUT,
)
for _ in range(90):
    if server.poll() is not None:
        print(log_path.read_text()[-2000:])
        raise SystemExit('The model server exited. See the log above.')
    try:
        urllib.request.urlopen(
            urllib.request.Request(f'http://127.0.0.1:{port}/health', headers={'Authorization': f'Bearer {token}'}),
            timeout=3,
        )
        break
    except Exception:
        time.sleep(2)
else:
    print(log_path.read_text()[-2000:])
    raise SystemExit('The model did not become ready.')
print('Model is ready on this GPU.')

cloudflared = Path('/content/cloudflared')
if not cloudflared.exists():
    subprocess.check_call([
        'curl', '-fsSL', '-o', str(cloudflared),
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
    ])
    cloudflared.chmod(0o755)
tunnel = subprocess.Popen(
    [str(cloudflared), 'tunnel', '--url', f'http://127.0.0.1:{port}'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
public = None
assert tunnel.stdout is not None
for line in tunnel.stdout:
    print(line, end='')
    match = re.search(r'https://[-0-9a-z]+\.trycloudflare\.com', line)
    if match:
        public = match.group(0)
        break
if not public:
    raise SystemExit('Cloudflare did not print a URL.')
import threading
def _drain():
    assert tunnel.stdout is not None
    for _rest in tunnel.stdout:
        pass
threading.Thread(target=_drain, daemon=True).start()
print()
print('Paste these lines into ai-interpreter/.env on the laptop, then restart python run.py:')
print('INTERPRETER_MODE=remote')
print(f'REMOTE_MODEL_URL={public}')
print(f'REMOTE_MODEL_TOKEN={token}')
print()
print('Leave this Colab runtime open. Stopping the runtime closes the tunnel.')